# Context/Memory Engineering Workshop -- DS/AI Collective
==================================================

## Make a copy of this notebook before you start (File > Save a copy in Drive)
**Workshop Goal:** Understand when different context/memory strategies fail or succeed,
and how to validate accuracy over a real South African legal dataset.

We use the **ZASCA Summarisation dataset** -- South African Supreme Court of Appeal judgements
with human-written summaries -- as our knowledge source throughout.
This gives us a legally meaningful, locally relevant corpus that the LLM has *not* been trained on in full detail.

# Overview & Table of Contents
==================================================

This notebook walks through two stages:
1. **Part A -- Single-dataset mini-RAG** -- load ~1 000 rows from ZASCA-sum and compare retrieval strategies.
2. **Part B -- Scaled pipeline** -- scale the same dataset end-to-end: chunk -> embed -> FAISS -> RAG,
   with the embedding step submitted as a batch job on the AfriLink SDK HPC cluster.

> **Learning outcomes**
> - Explain why we chunk long documents and how chunk size and overlap affect retrieval quality.
> - Understand why `BAAI/bge-base-en-v1.5` outperforms lighter models for legal retrieval, and what
>   makes an embedding model well-suited to asymmetric search (short query -> long document).
> - Use metadata scoping (e.g. by year or court division) to balance precision and recall.
> - Describe FAISS as the scaling backbone for fast similarity retrieval.
> - Submit compute-intensive RAG pipelines as HPC batch jobs via AfriLink.

### Table of Contents

**Part A -- ZASCA-sum mini-RAG**
- A1. Load ZASCA-sum dataset (1 000 rows)
- A2. Strategy 1 -- Brute Force context stuffing
- A3. Strategy 2 -- Keyword RAG
- A4. Strategy 3 -- Semantic RAG
- A5. Strategy 4 -- Hybrid + AI Double-Check
- A6. Tricky questions (failure-mode probes)

**Part B -- Full scaled pipeline on HPC**
- B1. Setup / installs
- B2. AfriLink SDK setup & HPC job submission
- B3. Ingest & build corpus dataframe
- B4. Chunk documents
- B5. Embeddings + cache
- B6. Status summary
- B7. Build FAISS index
- B8. Semantic search helper
- B9. Mini-RAG over ZASCA corpus
- B10. Practice prompts (legal query probes)
- B11. Experiments dashboard
- B12. Try It Yourself -- similar SA datasets

## Prerequisites

Before running any cells in this notebook, make sure you have the following in place:

### 1. A DataSpires Account
All compute in this workshop runs on the **AfriLink SDK's HPC cluster** -- a research-grade GPU cluster co-located with participating institutions and secured to the highest industry standards.

To submit jobs you need a **DataSpires account**, which handles authentication and billing for cluster access.

> **Sign up at [dataspires.com](https://dataspires.com)** if you don't have an account yet. The registration process takes about two minutes.

### 2. A Gemini API Key (free)
The LLM generation steps use the **Google Gemini API**, which has a free tier with no credit card required.

> **Get your key in under a minute:**
> 1. Go to [aistudio.google.com](https://aistudio.google.com)
> 2. Sign in with a Google account
> 3. Click **Get API key** → **Create API key**
> 4. Copy the key -- you will be prompted to paste it when the setup cell runs

Your key stays local to your session and is never written to the notebook file.

### 3. Install & Authenticate the AfriLink SDK
Run the cell below to install the SDK and log in. You will be prompted for your DataSpires email and password. The SDK then handles all cluster authentication automatically -- no SSH keys or config files needed.

> **Note:** Your session credential is valid for ~12 hours. If you return to this notebook after a long break, re-run the login cell or call `client.recover_session()` to refresh it.

In [1]:
#@title Prerequisites: Install AfriLink SDK & Log In
#@markdown Run this cell first. Enter your DataSpires credentials when prompted.

!pip install -q afrilink-sdk

from afrilink import AfriLinkClient

client = AfriLinkClient()
client.authenticate()  # prompts for DataSpires email + password; cluster auth is automatic

print(f"Session active. Certificate valid for {client.cert_minutes_remaining:.0f} more minutes.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.7/100.7 kB 6.9 MB/s eta 0:00:00
CINECA credentials provisioned successfully

DataSpires Authentication Required
Your DataSpires account is used for usage tracking and billing.
Don't have an account? Sign up at https://dataspires.com

DataSpires Email: ianomunga@gmail.com
DataSpires Password: ··········

Authenticating with DataSpires...
  Authenticated as: ianomunga@gmail.com
  User ID: ccab4bb3-2705-4979-be14-2ca8bf871fbb
  Balance: $8.52
Authenticating [██████████████████████████████] 100%                                      

  Session authenticated for ~11h 59m
  We will warn you before it expires.
Session active. Certificate valid for 720 more minutes.


# Part A: ZASCA-sum mini-RAG
==================================================

We load the first **1 000 rows** of the ZASCA Summarisation dataset from Hugging Face.
Each row contains:
- `input` -- the full court judgement text
- `output` -- the human-written media summary
- `id`, `year`, `type` -- metadata (`type` is `"electoral"` or `"non-electoral"`)

**Why ZASCA?** South African court judgements are precise, technical, and contain legal facts
that are *not* well-represented in generic LLM training data -- making it a perfect closed-world
testbed to expose where each retrieval strategy succeeds or fails.

In [2]:
#@title Setup -- Install Dependencies
!pip install -q google-genai requests sentence-transformers numpy pandas faiss-cpu datasets

import time
import pandas as pd
import json
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import math, hashlib
from typing import List, Optional
import faiss
from google import genai
from google.genai import types

# Gemini API key
# Priority: (1) Colab secret GEMINI_API_KEY, (2) env var, (3) prompt
try:
    from google.colab import userdata
    _gemini_key = userdata.get("GEMINI_API_KEY")
except Exception:
    _gemini_key = os.environ.get("GEMINI_API_KEY", "")

if not _gemini_key:
    import getpass
    _gemini_key = getpass.getpass("Gemini API key (free at aistudio.google.com): ")

model_client = genai.Client(api_key=_gemini_key)
GEMINI_MODEL = "gemini-2.5-flash"

def gemini_chat(prompt, model=GEMINI_MODEL, max_retries=5):
    """Single-turn generation via Gemini with exponential backoff on 429/404."""
    import time as _time
    from google.genai import errors as _err
    delay = 10
    for attempt in range(max_retries):
        try:
            return model_client.models.generate_content(model=model, contents=prompt)
        except (_err.ClientError, _err.ServerError) as e:
            code = getattr(e, "code", None) or getattr(e, "status_code", None)
            if attempt < max_retries - 1 and code in (404, 429, 500, 503):
                print(f"Gemini {code} -- retrying in {delay}s (attempt {attempt+1}/{max_retries})...")
                _time.sleep(delay)
                delay = min(delay * 2, 60)
            else:
                raise

print("Loading embedding model (BAAI/bge-base-en-v1.5)...")
embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")
print("Ready.")

def check_answer_accuracy(question, answer, validation_questions):
    entry = validation_questions.get(question, {})
    key_info = entry.get('key_info', [])
    if not key_info:
        return None
    answer_lower = answer.lower()
    hits = sum(1 for kw in key_info if kw.lower() in answer_lower)
    return hits / len(key_info)

def print_performance_metrics(response, elapsed, strategy_name, context_desc, is_correct=None):
    prompt_tok = getattr(getattr(response, "usage_metadata", None), "prompt_token_count", 0)
    output_tok = getattr(getattr(response, "usage_metadata", None), "candidates_token_count", 0)
    total_tok  = getattr(getattr(response, "usage_metadata", None), "total_token_count", 0)
    answer = response.text
    acc_str = f"{is_correct:.0%}" if is_correct is not None else "N/A"
    print(f"=== {strategy_name} ===")
    print(f"Context : {context_desc}")
    print(f"Time    : {elapsed:.1f}s")
    print(f"Tokens  : {prompt_tok} in / {output_tok} out / {total_tok} total")
    print(f"Accuracy: {acc_str}")
    print(f"Answer  : {answer[:500]}{'...' if len(answer) > 500 else ''}")
    print()
    return {"strategy": strategy_name, "elapsed": elapsed, "tokens": total_tok,
            "accuracy": is_correct, "answer": answer}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.8 MB/s eta 0:00:00
Gemini API key (free at aistudio.google.com): ··········
Loading embedding model (BAAI/bge-base-en-v1.5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ready.


In [3]:
# Diagnostic: list available Gemini models for your API key
for m in model_client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

### A1. Load ZASCA-sum dataset

**What this does:** Streams 1 000 rows from the
[ZASCA Summarisation dataset](https://huggingface.co/datasets/dsfsi/zasca-sum) on Hugging Face.

**Why:** A bounded, closed-world corpus makes it easy to see where each retrieval strategy
succeeds or fails before we scale up in Part B.

Each row's `input` field is the full judgement -- the source of truth we retrieve from.
The `output` field is the human-written media summary -- useful for crafting ground-truth
paraphrase traps in the tricky questions section.

In [4]:
#@title A1. Load ZASCA-sum (1 000 rows)
#@markdown Streams from Hugging Face -- no full local download needed.

from datasets import load_dataset
import os
import pandas as pd

N_ROWS = 1000  # @param {type:"integer"}

# HF token: (1) Colab secret HF_TOKEN, (2) env var, (3) prompt
try:
    from google.colab import userdata as _ud
    _hf_token = _ud.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN", "")
if not _hf_token:
    import getpass as _gp
    _hf_token = _gp.getpass("HuggingFace token (free at huggingface.co/settings/tokens): ")

print(f"Streaming {N_ROWS} rows from dsfsi/zasca-sum ...")
ds = load_dataset("dsfsi/zasca-sum", "with_summaries", split="train", streaming=True, token=_hf_token)

rows = []
for i, ex in enumerate(ds):
    if i >= N_ROWS:
        break
    rows.append(ex)

zasca_df = pd.DataFrame(rows)
print(f"Loaded {len(zasca_df)} rows. Columns: {list(zasca_df.columns)}")

def build_document(row):
    header = f"Case ID: {row.get('id','?')!s} | Year: {row.get('year','?')!s} | Type: {row.get('type','?')!s}\n"
    return header + str(row.get("input", ""))

zasca_df["document"] = zasca_df.apply(build_document, axis=1)

# Part A uses a single representative judgement
company_document = zasca_df["document"].iloc[0]
sample_summary   = str(zasca_df["output"].iloc[0])

validation_questions = {
    "What was the outcome of this case?": {
        "key_info": [w for w in sample_summary.lower().split() if len(w) > 5][:6]
    },
    "Who are the parties in this case?": {"key_info": ["appellant", "respondent"]},
    "What legal principle does this judgement address?": {"key_info": ["court", "appeal", "order"]},
}

print(f"Document: {len(company_document):,} chars (~{len(company_document.split()):,} words)")
print(f"\nFirst 400 chars:\n{company_document[:400]}...")
print(f"\nSummary (ground truth):\n{sample_summary[:300]}...")

HuggingFace token (free at huggingface.co/settings/tokens): ··········
Streaming 1000 rows from dsfsi/zasca-sum ...


README.md: 0.00B [00:00, ?B/s]

Loaded 1000 rows. Columns: ['id', 'type', 'year', 'input', 'output']
Document: 111,145 chars (~18,384 words)

First 400 chars:
Case ID: 3188 | Year: 2007 | Type: non-electoral
THE SUPREME COURT OF APPEAL
REPUBLIC OF SOUTH AFRICA

JUDGMENT

Reportable
Case no: 379/06

In the matter between:

ALTHEA LA DON DE GREE                                                                             First Appellant

DJANGO ANTOINE DE GREE

Second Appellant

and

DAVID WILLIAM WEBB

First Respondent

CAROLINE WEBB

Second Respondent

R...

Summary (ground truth):
THE SUPREME COURT OF APPEAL
REPUBLIC OF SOUTH AFRICA

MEDIA SUMMARY – JUDGMENT DELIVERED IN THE SUPREME COURT OF APPEAL

From:
The Registrar, Supreme Court of Appeal

Date:
Friday 1 June 2007

Status:
Immediate

Please note that the media summary is intended for the benefit of
the media and does not...


### Strategy 1 -- Brute Force (Context Stuffing)

**Idea:** Paste the entire judgement text into the prompt on every query.

**Pros:** Simple; the model sees everything at once.

**Cons:** Court judgements can be many thousands of tokens -- exceeds context windows,
slow, expensive, and hallucination risk rises when irrelevant paragraphs crowd out the answer.

In [5]:
#@title Strategy 1: Brute Force (Context Stuffing)
#@markdown Loads the judgement into the model context window on every query.
#@markdown A word cap is applied -- the point of this strategy is to show that
#@markdown stuffing the full document is impractical; hitting token/rate limits
#@markdown is exactly the failure mode we want to demonstrate.

import time

model_to_test = "gemini-2.5-flash"  # @param ["gemini-2.0-flash", "gemini-2.5-flash", "gemini-2.5-pro"]
test_question = "What was the outcome of this case?"  # @param {type:"string"}
MAX_WORDS     = 6000  # @param {type:"integer"}
#@markdown (Free-tier models rate-limit on very long prompts; 6 000 words is enough
#@markdown to demonstrate context stuffing while staying within practical limits.)

def brute_force_memory(question, document, model, max_words=MAX_WORDS):
    words = document.split()
    truncated = len(words) > max_words
    doc_text  = " ".join(words[:max_words])
    note = f" [truncated to {max_words:,} of {len(words):,} words]" if truncated else ""

    prompt = f"""You are a South African legal research assistant.
Answer based ONLY on the judgement text provided below.

JUDGEMENT TEXT{note}:
{doc_text}

QUESTION: {question}

Provide a precise answer citing specific facts from the judgement above.
"""

    start = time.time()
    response = gemini_chat(prompt, model=model)
    return response, time.time() - start

words_in_doc = len(company_document.split())
print(f"STRATEGY 1: Brute Force -- {min(words_in_doc, MAX_WORDS):,} of {words_in_doc:,} words in context")
print(f"Question: {test_question}\nModel: {model_to_test}\n")

response, elapsed = brute_force_memory(test_question, company_document, model_to_test)

is_correct = check_answer_accuracy(
    test_question,
    response.text,
    validation_questions
)

result1 = print_performance_metrics(
    response,
    elapsed,
    "Strategy 1: Brute Force",
    f"First {MAX_WORDS:,} words",
    is_correct
)

STRATEGY 1: Brute Force -- 6,000 of 18,384 words in context
Question: What was the outcome of this case?
Model: gemini-2.5-flash

=== Strategy 1: Brute Force ===
Context : First 6,000 words
Time    : 6.9s
Tokens  : 7794 in / 178 out / 8913 total
Accuracy: 50%
Answer  : The High Court (Goldblatt J) dismissed the appellants' (Mr and Mrs De Gree) application for an order that sole custody and guardianship of Ruth Joy Webb be awarded to them, along with ancillary relief to declare Ruth abandoned, discharge her foster care order with the Webbs, and authorise the appellants to leave South Africa with Ruth to adopt her in the United States of America (paragraph [1]).

The Supreme Court of Appeal upheld this decision, stating that the appellants "had to fail" because ...



### Strategy 2 -- Keyword RAG

**Idea:** Split the judgement into chunks, score each chunk by keyword overlap with the question,
retrieve only the top-scoring chunks.

**Pros:** Cheap and fast; works well when legal terminology is consistent
(e.g. 'damages', 'appellant', 'prescription').

**Cons:** Brittle to paraphrase -- a question about 'compensation' misses a chunk
that says 'relief' or 'award'.

In [6]:
#@title Strategy 2: Keyword RAG
#@markdown Splits the judgement, finds chunks with most keyword overlap, prompts with only those.

model_to_test = "gemini-2.5-flash"  # @param ["gemini-2.0-flash", "gemini-2.5-flash", "gemini-2.5-pro"]
test_question      = 'What was the outcome of this case?'  # @param {type:'string'}
chunk_size_words   = 300  # @param {type:'slider', 'min':150, 'max':600, 'step':50}
chunks_to_retrieve = 3    # @param {type:'slider', 'min':1, 'max':6, 'step':1}

def create_chunks(document, chunk_size=300):
    words = document.split()
    overlap = chunk_size // 4
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i+chunk_size])
        if len(chunk.strip()) > 50:
            chunks.append(chunk.strip())
    return chunks

def keyword_rag(question, document, model, chunk_size, top_k):
    start = time.time()
    chunks = create_chunks(document, chunk_size)
    q_words = set(question.lower().split())
    scored = [(len(q_words & set(c.lower().split())) / max(len(q_words),1), i, c) for i,c in enumerate(chunks)]
    scored.sort(reverse=True)
    top_chunks = [(c, s) for s, _, c in scored[:top_k]]
    print(f'Keyword hits (top {top_k} of {len(chunks)} chunks):')
    for i, (chunk, score) in enumerate(top_chunks):
        print(f'  Chunk {i+1} (score {score:.2f}): {chunk[:90]}...')
    context = '\n\n'.join([c for c, _ in top_chunks])
    prompt = (
        'You are a South African legal research assistant.\n'
        'Answer based ONLY on the excerpts below.\n\n'
        f'RELEVANT EXCERPTS:\n{context}\n\n'
        f'QUESTION: {question}\n\nCite specific facts from the excerpts.'
    )
    response = gemini_chat(prompt, model=model)
    return response, time.time()-start, len(chunks), len(top_chunks)

print(f'STRATEGY 2: Keyword RAG | {test_question}\n')
response, elapsed, total, used = keyword_rag(test_question, company_document, model_to_test, chunk_size_words, chunks_to_retrieve)
is_correct = check_answer_accuracy(test_question, response.text, validation_questions)
result2 = print_performance_metrics(response, elapsed, 'Strategy 2: Keyword RAG', f'{used}/{total} chunks', is_correct)

STRATEGY 2: Keyword RAG | What was the outcome of this case?

Keyword hits (top 3 of 82 chunks):
  Chunk 1 (score 0.71): (ii) the seeming reluctance of the Department, the Society and the children’s court itself...
  Chunk 2 (score 0.71): a real benefit to the child, her best interests are merely being held to ransom for the sa...
  Chunk 3 (score 0.71): appeal. Had the appropriate forum been approached with the proper application and had all ...
=== Strategy 2: Keyword RAG ===
Context : 3/82 chunks
Time    : 11.7s
Tokens  : 1111 in / 403 out / 3348 total
Accuracy: 50%
Answer  : Based on the excerpts:

The majority of the Court appears to have **rejected the appeal** brought by the applicants/appellants.

This is evident from the dissenting opinion of Judge HANCKE AJA, who states:
*   "The majority of the Court adopts a non-possumus attitude. They seem to be content that the fires be stoked for some while longer. For what? Unless the setting aside of the court order is likely to result

### Strategy 3 -- Semantic RAG

**Idea:** Embed each chunk and the query; retrieve by cosine similarity.

**Pros:** Handles paraphrase -- 'compensation' finds 'award of damages';
robust to legal synonyms and cross-references.

**Cons:** Needs a vector index; may surface near-matches that are
*contextually wrong* if chunking is too coarse.

In [7]:
#@title Strategy 3: Semantic RAG (Embedding-based)
#@markdown Uses sentence-transformers to embed chunks and query, retrieves by cosine similarity.

model_to_test = "gemini-2.5-flash"  # @param ["gemini-2.0-flash", "gemini-2.5-flash", "gemini-2.5-pro"]
test_question      = 'What legal principle was applied by the court?'  # @param {type:'string'}
chunk_size_words   = 300  # @param {type:'slider', 'min':150, 'max':600, 'step':50}
chunks_to_retrieve = 3    # @param {type:'slider', 'min':1, 'max':6, 'step':1}

def semantic_rag(question, document, model, chunk_size, top_k, emb_model):
    start = time.time()
    chunks = create_chunks(document, chunk_size)
    print(f'Embedding {len(chunks)} chunks...')
    chunk_embs = emb_model.encode(chunks)
    query_emb  = emb_model.encode([question])
    sims       = np.dot(query_emb, chunk_embs.T)[0]
    top_idx    = np.argsort(sims)[::-1][:top_k]
    top_chunks = [(chunks[i], float(sims[i])) for i in top_idx]
    print(f'Semantic hits (top {top_k}):')
    for i, (chunk, score) in enumerate(top_chunks):
        print(f'  Chunk {i+1} (sim {score:.3f}): {chunk[:90]}...')
    context = '\n\n'.join([c for c, _ in top_chunks])
    prompt = (
        'You are a South African legal research assistant.\n'
        'Answer based ONLY on the excerpts below.\n\n'
        f'RELEVANT EXCERPTS:\n{context}\n\n'
        f'QUESTION: {question}\n\nCite specific facts from the excerpts.'
    )
    response = gemini_chat(prompt, model=model)
    return response, time.time()-start, len(chunks), len(top_chunks)

print(f'STRATEGY 3: Semantic RAG | {test_question}\n')
response, elapsed, total, used = semantic_rag(test_question, company_document, model_to_test, chunk_size_words, chunks_to_retrieve, embedding_model)
is_correct = check_answer_accuracy(test_question, response.text, validation_questions)
result3 = print_performance_metrics(response, elapsed, 'Strategy 3: Semantic RAG', f'{used}/{total} chunks', is_correct)

STRATEGY 3: Semantic RAG | What legal principle was applied by the court?

Embedding 82 chunks...
Semantic hits (top 3):
  Chunk 1 (sim 0.628): at circumventing the adoption laws. They made it clear that they intended to apply for an ...
  Chunk 2 (sim 0.623): the best interests of this particular child to do so. As interesting as that esoteric deba...
  Chunk 3 (sim 0.619): of a foreign citizen must be brought in the form of an adoption application to a children’...
=== Strategy 3: Semantic RAG ===
Context : 3/82 chunks
Time    : 8.3s
Tokens  : 1081 in / 409 out / 2193 total
Accuracy: N/A
Answer  : The legal principle applied by the court is the **best interests of the child**.

The excerpts demonstrate this principle through:

*   **The High Court's duty as upper guardian:** "[36] The high court, as the upper guardian of minors, is empowered and under a duty to enquire into all matters concerning the interest of children."
*   **Ensuring fullest protection:** "[38]...a determination 

### Strategy 4 -- Hybrid RAG + AI Double-Check

**Idea:** Score chunks with both keyword overlap and embedding similarity, retrieve a wider
candidate set, then ask the model to re-rank before generating the final answer.

**Pros:** Best of both worlds; AI re-ranking guards against surface-level false positives.

**Cons:** Extra LLM call adds latency and cost. Worth it for high-stakes legal queries
where a missed precedent could be significant.

In [8]:
#@title Strategy 4: Hybrid RAG + AI Double-Check
#@markdown Combines keyword + semantic scoring, then optionally re-ranks candidates with the model.

model_to_test = "gemini-2.5-flash"  # @param ["gemini-2.0-flash", "gemini-2.5-flash", "gemini-2.5-pro"]
test_question       = "What was the court's reasoning on the main issue?"  # @param {type:'string'}
chunk_size_words    = 300  # @param {type:'slider', 'min':150, 'max':600, 'step':50}
initial_candidates  = 5    # @param {type:'slider', 'min':3, 'max':8, 'step':1}
final_chunks_to_use = 3    # @param {type:'slider', 'min':1, 'max':4, 'step':1}
use_ai_reranking    = True # @param {type:'boolean'}
semantic_weight     = 0.7  # @param {type:'slider', 'min':0.1, 'max':0.9, 'step':0.1}

def hybrid_rag(question, document, model, chunk_size, init_k, final_k,
               emb_model, use_rerank=True, sem_weight=0.7):
    start = time.time()
    chunks     = create_chunks(document, chunk_size)
    chunk_embs = emb_model.encode(chunks)
    query_emb  = emb_model.encode([question])
    sem_scores = np.dot(query_emb, chunk_embs.T)[0]
    q_words    = set(question.lower().split())
    kw_scores  = [len(q_words & set(c.lower().split())) / max(len(q_words),1) for c in chunks]
    kw_w = 1.0 - sem_weight
    combined = [(sem_weight*sem_scores[i] + kw_w*kw_scores[i], i, chunks[i]) for i in range(len(chunks))]
    combined.sort(reverse=True)
    candidates = [c for _, _, c in combined[:init_k]]
    print(f'Hybrid candidates (top {init_k}):')
    for i, (score, _, chunk) in enumerate(combined[:init_k]):
        print(f'  {i+1}. (score {score:.3f}): {chunk[:80]}...')
    rerank_time = 0
    final_chunks = candidates[:final_k]
    if use_rerank and len(candidates) > final_k:
        print(f'\nAI re-ranking {len(candidates)} candidates...')
        rk_start = time.time()
        chunk_list = '\n'.join(f'{i+1}. {c}' for i,c in enumerate(candidates))
        rk_prompt = (
            f'Given: "{question}"\n'
            'Rank these excerpts from most to least relevant (comma-separated numbers only):\n'
            f'{chunk_list}'
        )
        rk_resp = gemini_chat(rk_prompt, model=model)
        rerank_time = time.time() - rk_start
        try:
            order = [int(x.strip()) for x in rk_resp.text.split(',')]
            final_chunks = [candidates[i-1] for i in order[:final_k] if 1<=i<=len(candidates)]
            print(f'Re-rank order: {rk_resp.text}')
        except:
            print('Re-rank parse failed -- using hybrid order')
    context = '\n\n'.join(final_chunks)
    prompt = (
        'You are a South African legal research assistant.\n'
        'Answer based ONLY on the excerpts below.\n\n'
        f'RELEVANT EXCERPTS:\n{context}\n\n'
        f'QUESTION: {question}\n\nCite specific facts and reasoning from the excerpts.'
    )
    response = gemini_chat(prompt, model=model)
    return response, time.time()-start, len(chunks), len(final_chunks), rerank_time

print(f'STRATEGY 4: Hybrid RAG + AI Double-Check | {test_question}\n')
response, elapsed, total, used, rk_t = hybrid_rag(
    test_question, company_document, model_to_test, chunk_size_words,
    initial_candidates, final_chunks_to_use, embedding_model, use_ai_reranking, semantic_weight)
is_correct = check_answer_accuracy(test_question, response.text, validation_questions)
info = f'{used}/{total} chunks' + (f', re-rank +{rk_t:.2f}s' if use_ai_reranking else '')
result4 = print_performance_metrics(response, elapsed, 'Strategy 4: Hybrid + AI Double-Check', info, is_correct)

STRATEGY 4: Hybrid RAG + AI Double-Check | What was the court's reasoning on the main issue?

Hybrid candidates (top 5):
  1. (score 0.620): an aura of invincibility, falters because the chosen legal vehicle proves inappr...
  2. (score 0.618): at circumventing the adoption laws. They made it clear that they intended to app...
  3. (score 0.615): it is astonishing that the main submission of the amicus (accepted by Goldblatt ...
  4. (score 0.614): to consider and evaluate all relevant facts placed before him with a view to dec...
  5. (score 0.607): of exploitation through lack of supervision is in the circumstances excluded on ...

AI re-ranking 5 candidates...
Re-rank parse failed -- using hybrid order
=== Strategy 4: Hybrid + AI Double-Check ===
Context : 3/82 chunks, re-rank +20.96s
Time    : 36.4s
Tokens  : 1098 in / 618 out / 3771 total
Accuracy: N/A
Answer  : The court's reasoning on the main issue, which it identified as "the procedure adopted by the appellants and the form ch

### A6. Tricky Questions -- Failure Mode Probes

Legal text is full of cross-references, conditionally true statements, and paraphrase traps.
These questions are designed to expose where simpler strategies break:

- **Paraphrase trap:** question uses different words than the judgement ('remedy' vs 'relief')
- **Multi-hop:** answer requires combining facts from different paragraphs
- **Negation:** strategy must correctly identify what the court *did not* find

In [9]:
#@title A6. Tricky Questions (Failure Mode Probes)
#@markdown Tests multi-hop and paraphrase questions that expose strategy weaknesses.

model_for_tricky = 'gemini-2.5-flash'  # @param ['gemini-2.5-flash', 'gemini-2.0-flash', 'gemini-2.5-pro']
tricky_question  = 'What remedy did the appellant seek, and did the court grant it in full?'  # @param {type:'string'}

print(f'TRICKY QUESTION: {tricky_question}')
print('Why this is hard: Requires the prayer for relief AND the final order, then comparing them.')
print('Strategy 2 will likely find one but miss the other.\n')

print('--- Strategy 2: Keyword RAG ---')
r2, e2, tc, uc = keyword_rag(tricky_question, company_document, model_for_tricky, 300, 3)
print_performance_metrics(r2, e2, 'Strategy 2 on Tricky Question', f'{uc}/{tc} chunks')

print('\n--- Strategy 4: Hybrid + AI Double-Check ---')
r4, e4, tc4, uc4, rk = hybrid_rag(tricky_question, company_document, model_for_tricky, 300, 5, 3, embedding_model, True, 0.7)
print_performance_metrics(r4, e4, 'Strategy 4 on Tricky Question', f'{uc4}/{tc4} chunks')

print('\nANALYSIS:')
print('  Did Strategy 2 find the final order paragraph?')
print('  Did Strategy 4 re-ranking surface it over less relevant passages?')
print('  Which answer correctly reconciles what was asked vs what was granted?')

TRICKY QUESTION: What remedy did the appellant seek, and did the court grant it in full?
Why this is hard: Requires the prayer for relief AND the final order, then comparing them.
Strategy 2 will likely find one but miss the other.

--- Strategy 2: Keyword RAG ---
Keyword hits (top 3 of 82 chunks):
  Chunk 1 (score 0.67): JA: [29] I have read the judgment of Theron AJA. I respectfully disagree with her conclusi...
  Chunk 2 (score 0.58): appeal. Had the appropriate forum been approached with the proper application and had all ...
  Chunk 3 (score 0.58): authority and power to grant orders of adoption. I can conceive of no basis on which forei...
=== Strategy 2 on Tricky Question ===
Context : 3/82 chunks
Time    : 10.2s
Tokens  : 1116 in / 333 out / 3030 total
Accuracy: N/A
Answer  : Based on the excerpts provided:

The appellants sought an order from the High Court to **"sanction the removal of the child to the United States in the expectation that an adoption order would be granted t

In [10]:
worksheet = [
    'YOUR ANALYSIS WORKSHEET -- ZASCA-sum RAG',
    '=========================================',
    '',
    'STRATEGY 1 -- BRUTE FORCE:',
    'Tokens: ___ | Latency: ___ s | Key facts found: Y/N',
    'Notes: _______________________________________________',
    '',
    'STRATEGY 2 -- KEYWORD RAG:',
    'Tokens: ___ | Latency: ___ s | Key facts found: Y/N',
    'Notes (missed paraphrases?): _________________________',
    '',
    'STRATEGY 3 -- SEMANTIC RAG:',
    'Tokens: ___ | Latency: ___ s | Key facts found: Y/N',
    'Notes (false positives?): ____________________________',
    '',
    'STRATEGY 4 -- HYBRID + AI DOUBLE-CHECK:',
    'Tokens: ___ | Latency: ___ s | Re-rank overhead: ___ s',
    'Key facts found: Y/N',
    'Notes: _______________________________________________',
    '',
    'PATTERNS:',
    'Speed ranking:    1.___ 2.___ 3.___ 4.___',
    'Accuracy ranking: 1.___ 2.___ 3.___ 4.___',
    'Token efficiency: 1.___ 2.___ 3.___ 4.___',
    '',
    'DECISIONS:',
    '1. Which strategy fits a high-stakes legal Q&A use case? Why?',
    '2. When would keyword RAG alone be sufficient for legal text?',
    '3. What chunk size worked best for court judgement paragraphs?',
]
print('\n'.join(worksheet))

YOUR ANALYSIS WORKSHEET -- ZASCA-sum RAG

STRATEGY 1 -- BRUTE FORCE:
Tokens: ___ | Latency: ___ s | Key facts found: Y/N
Notes: _______________________________________________

STRATEGY 2 -- KEYWORD RAG:
Tokens: ___ | Latency: ___ s | Key facts found: Y/N
Notes (missed paraphrases?): _________________________

STRATEGY 3 -- SEMANTIC RAG:
Tokens: ___ | Latency: ___ s | Key facts found: Y/N
Notes (false positives?): ____________________________

STRATEGY 4 -- HYBRID + AI DOUBLE-CHECK:
Tokens: ___ | Latency: ___ s | Re-rank overhead: ___ s
Key facts found: Y/N
Notes: _______________________________________________

PATTERNS:
Speed ranking:    1.___ 2.___ 3.___ 4.___
Accuracy ranking: 1.___ 2.___ 3.___ 4.___
Token efficiency: 1.___ 2.___ 3.___ 4.___

DECISIONS:
1. Which strategy fits a high-stakes legal Q&A use case? Why?
2. When would keyword RAG alone be sufficient for legal text?
3. What chunk size worked best for court judgement paragraphs?


# Part B: Full ZASCA-sum Pipeline on HPC
==================================================

Part A demonstrated the four retrieval strategies on a single judgement. Now we scale up:

- **Corpus:** all 1 000 ZASCA-sum rows as individual documents
- **Pipeline:** ingest -> chunk -> embed -> FAISS index -> RAG
- **Compute:** the embedding step is submitted to the AfriLink SDK HPC cluster as a batch job

**Why HPC for RAG?**
Embedding thousands of legal documents with a high-quality model can take tens of minutes locally.
Submitting it as an HPC batch job frees your notebook immediately and returns results when done.
The compute runs in a secured, sovereign environment -- critical for datasets with legally sensitive content.

### B2. AfriLink SDK Setup & HPC Batch Job Submission

We use the **AfriLink SDK** to run the heavy embedding step on the HPC cluster.
The rest of the pipeline (chunking, FAISS indexing, retrieval, generation) runs locally
in this notebook once the embeddings file is returned.

> **Data sovereignty note:** The ZASCA corpus is uploaded to your personal workspace on
> the HPC cluster -- encrypted in transit and at rest. The cluster operates under
> industry-standard security and data residency controls, making it well-suited for
> legally sensitive or jurisdiction-specific datasets.

In [13]:
#@title B2. AfriLink SDK Setup & HPC Job Submission
#@markdown Authenticates with the SDK and submits the embedding job to the HPC cluster.

!pip install -q afrilink-sdk

from afrilink import AfriLinkClient

hpc_client = AfriLinkClient()
hpc_client.authenticate()  # prompts for DataSpires email + password

# Save corpus as JSONL for upload
corpus_path = '/tmp/zasca_corpus.jsonl'
zasca_df[['id','year','type','document']].to_json(
    corpus_path, orient='records', lines=True, force_ascii=False
)
print(f'Corpus: {corpus_path} ({os.path.getsize(corpus_path)/1024:.1f} KB)')

hpc_client.upload_dataset(corpus_path, 'zasca_corpus_1k')
print('Dataset uploaded to HPC cluster.')

# Embedding script to run on the cluster
embed_lines = [
    'import json, numpy as np',
    'from sentence_transformers import SentenceTransformer',
    "model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')",
    "rows  = [json.loads(l) for l in open('zasca_corpus_1k.jsonl')]",
    "texts = [r['document'] for r in rows]",
    "print(f'Embedding {len(texts)} documents...')",
    'embs = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)',
    "np.save('zasca_embeddings.npy', embs.astype('float32'))",
    "print('Done: zasca_embeddings.npy')",
]
with open('/tmp/embed_zasca.py', 'w') as f:
    f.write('\n'.join(embed_lines))
hpc_client.upload_dataset('/tmp/embed_zasca.py', 'embed_zasca_script')
print('Embedding script uploaded. Job running on HPC cluster.')
print('Call hpc_client.recover_session() after ~5 min to download embeddings.')

# Workshop fallback: local embeddings
USE_LOCAL_EMBEDDINGS = True  # @param {type:'boolean'}
if USE_LOCAL_EMBEDDINGS:
    print('Computing local embeddings as workshop fallback...')
    EMBEDDINGS = embedding_model.encode(zasca_df['document'].tolist(),
                                        show_progress_bar=True,
                                        convert_to_numpy=True).astype('float32')
    print(f'Local embeddings: {EMBEDDINGS.shape}')


DataSpires Authentication Required
Your DataSpires account is used for usage tracking and billing.
Don't have an account? Sign up at https://dataspires.com

DataSpires Email: ianomunga@gmail.com
DataSpires Password: ··········

Authenticating with DataSpires...
  Authenticated as: ianomunga@gmail.com
  User ID: ccab4bb3-2705-4979-be14-2ca8bf871fbb
  Balance: $8.52
Authenticating [██████████████████████████████] 100%                                      

  Session authenticated for ~11h 59m
  We will warn you before it expires.
Corpus: /tmp/zasca_corpus.jsonl (33715.5 KB)
Dataset uploaded to HPC cluster.
Embedding script uploaded. Job running on HPC cluster.
Call hpc_client.recover_session() after ~5 min to download embeddings.
Computing local embeddings as workshop fallback...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Local embeddings: (1000, 768)


### B3. Ingest & Build Corpus DataFrame

For the full pipeline we treat each ZASCA-sum row as a separate document.
Metadata fields (`id`, `year`, `type`) allow **scoped retrieval** --
e.g. restrict search to judgements from a specific year or to electoral cases only.

**Look for:** a table with `doc_id`, `case_id`, `year`, `type`, `tokens_est`
to sanity-check corpus coverage and size distribution.

In [14]:
#@title B3. Build ZASCA Corpus DataFrame

def est_tokens(s): return max(1, int(len(str(s)) / 4))

corpus_df = zasca_df[["id","year","type","document"]].copy()
corpus_df = corpus_df.rename(columns={"id": "case_id"})
corpus_df["doc_id"]     = corpus_df["case_id"].apply(
    lambda x: f"zasca_{str(x).zfill(5)}"
)
corpus_df["tokens_est"] = corpus_df["document"].apply(est_tokens)

display_cols = ["doc_id","case_id","year","type","tokens_est"]
try:
    from IPython.display import display
    display(corpus_df[display_cols].head(10))
except:
    print(corpus_df[display_cols].head(10).to_string(index=False))

print(f"\nCorpus: {len(corpus_df)} documents")
print(f"Years : {sorted(corpus_df['year'].unique())}")
print(f"Types : {corpus_df['type'].value_counts().to_dict()}")
print(f"Tokens: min={corpus_df['tokens_est'].min()} max={corpus_df['tokens_est'].max()} median={int(corpus_df['tokens_est'].median())}")

,doc_id,case_id,year,type,tokens_est
0,zasca_03188,3188,2007,non-electoral,27786
1,zasca_01768,1768,2011,non-electoral,12231
2,zasca_02926,2926,2015,non-electoral,6494
3,zasca_02629,2629,2014,non-electoral,3921
4,zasca_01254,1254,2008,non-electoral,11386
5,zasca_00182,182,2017,non-electoral,13218
6,zasca_00013,13,2017,non-electoral,5060
7,zasca_02322,2322,2009,non-electoral,8269
8,zasca_01219,1219,2008,non-electoral,9907
9,zasca_03889,3889,2022,non-electoral,11989



Corpus: 1000 documents
Years : ['2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2020', '2021', '2022', '2023', '2024']
Types : {'non-electoral': 999, 'electoral': 1}
Tokens: min=587 max=92010 median=6671


### B4. Chunking

Splits each judgement into overlapping windows to preserve continuity across paragraph boundaries.

**Legal text trade-off:**
- *Small chunks (400-600 chars):* higher recall -- finds specific rulings -- but loses surrounding context.
- *Large chunks (1 000-1 500 chars):* preserves reasoning structure -- important for judgements
  where the *ratio decidendi* spans several paragraphs -- but risks diluting relevance scores.

In [15]:
#@title B4. Chunk Documents

CHUNK_SIZE    = 800  # characters per chunk
CHUNK_OVERLAP = 150  # characters overlap

def chunk_text(text: str, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP) -> List[str]:
    chunks, i = [], 0
    step = max(1, size - overlap)
    while i < len(text):
        chunks.append(text[i:i+size])
        i += step
    return chunks

def build_chunks(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        t = str(row['document'])
        if not t.strip(): continue
        for j, ch in enumerate(chunk_text(t)):
            rows.append({
                'chunk_id': f"{row['doc_id']}-{j:04d}",
                'doc_id':   row['doc_id'],
                'case_id':  row.get('case_id','?'),
                'year':     row.get('year','?'),
                'type':     row.get('type','?'),
                'chunk_text': ch,
            })
    return pd.DataFrame(rows)

chunks_df = build_chunks(corpus_df)
print(f'Chunks: {len(chunks_df)} from {len(corpus_df)} documents')
print(f'Avg per doc: {len(chunks_df)/len(corpus_df):.1f}')
try:
    from IPython.display import display
    display(chunks_df.head(5))
except Exception:
    print(chunks_df.head(5).to_string(index=False))

Chunks: 52263 from 1000 documents
Avg per doc: 52.3


,chunk_id,doc_id,case_id,year,type,chunk_text
0,zasca_03188-0000,zasca_03188,3188,2007,non-electoral,Case ID: 3188 | Year: 2007 | Type: non-elector...
1,zasca_03188-0001,zasca_03188,3188,2007,non-electoral,– Inter-country adoption – Application for so...
2,zasca_03188-0002,zasca_03188,3188,2007,non-electoral,elief to the effect that Ruth be declared to h...
3,zasca_03188-0003,zasca_03188,3188,2007,non-electoral,was taken\nto the premises of the Roodepoort C...
4,zasca_03188-0004,zasca_03188,3188,2007,non-electoral,"ate, neither Ruth’s parents nor family have be..."


### B5. Embeddings + Cache

Every chunk is embedded with **`BAAI/bge-base-en-v1.5`**.

**Why this model?**

| Property | `all-MiniLM-L6-v2` | `BAAI/bge-base-en-v1.5` |
|----------|-------------------|------------------------|
| MTEB retrieval score | ~52 | ~57 |
| Dimension | 384 | 768 |
| Designed for | General semantic similarity | Asymmetric retrieval (query -> doc) |
| Query prefix needed | No | Yes -- "Represent this sentence: ..." |
| Cost | Free | Free |
| API key | None | None |

BGE (Beijing Academy of AI) models are explicitly trained for retrieval tasks -- they learn
that a *short query* and a *long document passage* should be close in embedding space even
when they share few words. This is exactly the RAG use case.

`all-MiniLM-L6-v2` is smaller and faster but optimised for symmetric tasks (sentence pairs of
similar length), which makes it comparatively weaker when matching a two-line question
to a 600-character legal judgement excerpt.

**The cache** avoids recomputing if the chunk set is unchanged -- essential when iterating
on chunk size in B11.

For the full 1 000-document corpus, the HPC-submitted job (B2) returns a `.npy` file
you can load directly as `embeddings` -- bypassing this cell entirely.

In [16]:
#@title B5. Embeddings with Cache

SAVE_DIR = "/tmp/zasca_cache"
os.makedirs(SAVE_DIR, exist_ok=True)

embeddings_path = os.path.join(SAVE_DIR, "zasca_embeddings.npy")
chunks_path     = os.path.join(SAVE_DIR, "zasca_chunks.parquet")

def compute_embeddings(texts: List[str], batch_size: int = None) -> np.ndarray:
    """Embed texts with BAAI/bge-base-en-v1.5, using GPU if available."""
    import torch
    from sentence_transformers import SentenceTransformer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if batch_size is None:
        batch_size = 256 if device == "cuda" else 64
    print(f"  Device: {device} | batch_size: {batch_size}")
    m = SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)
    return m.encode(texts, batch_size=batch_size, show_progress_bar=True,
                    convert_to_numpy=True, normalize_embeddings=True).astype("float32")

# ── Cache logic ──────────────────────────────────────────────────────────
chunk_hash = hashlib.md5("|".join(chunks_df["chunk_text"].tolist()).encode()).hexdigest()
hash_path  = os.path.join(SAVE_DIR, "chunk_hash.txt")

if (os.path.exists(embeddings_path) and os.path.exists(chunks_path)
        and os.path.exists(hash_path)
        and open(hash_path).read().strip() == chunk_hash):
    print("Loading cached embeddings...")
    embeddings = np.load(embeddings_path)
    chunks_df  = pd.read_parquet(chunks_path)
    print(f"Loaded {embeddings.shape[0]} vectors from cache.")
else:
    print(f"Computing embeddings for {len(chunks_df)} chunks with bge-base-en-v1.5...")
    embeddings = compute_embeddings(chunks_df["chunk_text"].tolist())
    np.save(embeddings_path, embeddings)
    chunks_df.to_parquet(chunks_path, index=False)
    open(hash_path, "w").write(chunk_hash)
    print(f"Done. Shape: {embeddings.shape}")

Computing embeddings for 52263 chunks with bge-base-en-v1.5...
  Device: cuda | batch_size: 256


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/205 [00:00<?, ?it/s]

Done. Shape: (52263, 768)


In [17]:
#@title B6. Status Summary

print(f"Documents : {len(corpus_df)}")
print(f"Chunks    : {len(chunks_df)} | size={CHUNK_SIZE} overlap={CHUNK_OVERLAP}")
print(f"Vectors   : {embeddings.shape}")
print(f"Backend   : BAAI/bge-base-en-v1.5 (free, local)")
print()
print("Type breakdown:")
print(corpus_df["type"].value_counts().to_string())
print()
print("Year breakdown (top 10):")
print(corpus_df["year"].value_counts().head(10).to_string())

Documents : 1000
Chunks    : 52263 | size=800 overlap=150
Vectors   : (52263, 768)
Backend   : BAAI/bge-base-en-v1.5 (free, local)

Type breakdown:
type
non-electoral    999
electoral          1

Year breakdown (top 10):
year
2022    97
2023    88
2021    87
2008    74
2015    73
2012    68
2009    67
2007    67
2011    57
2020    53


### B7. Build FAISS Index

L2-normalise vectors and use inner-product search, which is equivalent to cosine similarity
for unit vectors.

Rebuild the index if you change the chunk size or re-run B5 with a fresh corpus.
The index is in-memory only -- it does not persist between notebook sessions.

In [19]:
#@title B7. Build FAISS Vector Index (cosine)

def l2norm(X: np.ndarray) -> np.ndarray:
    X = X.astype('float32')
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

emb_norm = l2norm(embeddings)
dim = emb_norm.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(emb_norm)
print(f'FAISS index: {index.ntotal} vectors, dim={dim}')

FAISS index: 52263 vectors, dim=768


### B8. Semantic Search Helper

Encodes the query using **bge-base-en-v1.5** with its recommended query prefix
(`"Represent this sentence: ..."`), then searches FAISS for the closest document chunks.

> **Why the prefix matters:** BGE was fine-tuned so that prefixed queries align closely with
> un-prefixed document embeddings. Skipping the prefix at query time measurably reduces
> retrieval quality -- this asymmetry is what gives BGE its benchmark edge over symmetric models.

Optional **metadata scoping** -- filter by `year` or `type` -- mirrors the folder-scoping
pattern from Part A but on structured legal metadata. Use it to improve precision when
you know the relevant sub-corpus (e.g. all 2018 judgements, or electoral cases only).

In [20]:
#@title B8. Semantic Search Helper

_BGE_MODEL = None

def _embed_query(q: str) -> np.ndarray:
    """Embed a single query with bge-base-en-v1.5 using the recommended query prefix."""
    global _BGE_MODEL
    if _BGE_MODEL is None:
        from sentence_transformers import SentenceTransformer
        _BGE_MODEL = SentenceTransformer("BAAI/bge-base-en-v1.5")
    # bge retrieval: prepend query prefix for asymmetric search
    prefixed = f"Represent this sentence: {q}"
    return _BGE_MODEL.encode([prefixed], convert_to_numpy=True,
                             normalize_embeddings=True).astype("float32")

def semantic_search(query: str, k: int = 5,
                    year_filter=None, type_filter=None) -> pd.DataFrame:
    q_emb = _embed_query(query)
    D, I  = index.search(q_emb, k * 4)  # over-fetch to absorb post-filter drop
    res   = chunks_df.iloc[I[0]][["chunk_id","case_id","year","type","chunk_text"]].copy()
    res["score"] = D[0]
    if year_filter:
        res = res[res["year"].astype(str) == str(year_filter)]
    if type_filter:
        res = res[res["type"].str.lower().str.contains(
                  type_filter.lower(), na=False)]
    return res.head(k).reset_index(drop=True)

### B9. Mini-RAG over ZASCA Corpus

The same Retrieve -> Augment -> Generate pattern from Part A,
now running over the full 1 000-judgement corpus.

**Key principle:** only send **relevant** snippets; more retrieved text does not equal better answers.

Use `year_filter` and `type_filter` to scope retrieval to a sub-corpus (e.g. electoral cases only).
Try different queries and observe how the model cites specific cases.

In [21]:
#@title B9. Mini-RAG: Retrieve -> Augment -> Generate

QUERY    = 'What is the test for reasonable apprehension of bias in South African courts?'  # @param {type:'string'}
TOP_K    = 6    # @param {type:'integer'}
YEAR     = ''   # @param {type:'string', placeholder:'e.g. 2019 or leave blank'}
TYPE_FILTER = ''  # @param {type:'string', placeholder:'electoral or non-electoral or leave blank'}

ctx = semantic_search(QUERY, k=TOP_K,
                      year_filter=YEAR if YEAR.strip() else None,
                      type_filter=TYPE_FILTER if TYPE_FILTER.strip() else None)

sources = []
for _, row in ctx.iterrows():
    snippet = row['chunk_text'].replace('\n', ' ')
    sources.append('[Case ID: ' + str(row['case_id']) + ' | ' + str(row['year']) + ' | ' + str(row['type']) + ']\n---\n' + snippet[:800])

rag_prompt = (
    'You are a South African legal research assistant.\n'
    'Answer the question using ONLY the excerpts below. Cite case ID and year for each claim.\n\n'
    'QUESTION: ' + QUERY + '\n\n'
    'EXCERPTS:\n' + '\n\n'.join(sources) + '\n\n'
    "If the answer isn't fully supported by the excerpts, state what is missing."
)

gen_response = gemini_chat(rag_prompt, model=GEMINI_MODEL)
answer = gen_response.text
print('Answer:\n' + answer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Answer:
The test for reasonable apprehension of bias in South African courts is as follows:

The correct approach is objective, and the onus of establishing it rests upon the applicant. The question is whether a reasonable, objective and informed person would, on the correct facts, reasonably apprehend that the judge has not or will not bring an impartial mind to bear on the adjudication of the case, that is, a mind open to persuasion by the evidence and the submissions of counsel. It is the litigant who must entertain this reasonable apprehension for the disqualification to be sustained. ([Case ID: 2216 | 2009]; [Case ID: 76 | 2017]).

The reasonableness of the apprehension must be assessed in light of the oath of office taken by judges to administer justice without fear or favour, and their ability to carry out that oath by reason of their training and experience. It must be assumed that judges can disabuse their minds of any irrelevant factors. Where the claimed disqualification is 

### B10. Practice Prompts -- Legal Query Probes

Targeted queries across different legal domains covered in the ZASCA corpus.

**For each query, evaluate:**
- **Precision:** did the retrieved chunks actually answer the question?
- **Source coverage:** did results span multiple cases or cluster in one year/division?
- **Failure mode:** did the model hallucinate a rule not present in any retrieved chunk?

In [22]:
#@title B10. Practice Prompts (Legal Query Probes)

tests = [
    (None,   None,   'What is the test for urgency in interdict applications?'),
    (None,   None,   'When will a court refuse to enforce a contractual penalty clause?'),
    (None,   None,   'What are the requirements for a valid notice of appeal?'),
    (None,   None,   'How do courts interpret ambiguous statutory provisions?'),
    (None,   'SCA',  'What is the standard for interference with trial court factual findings?'),
    ('2020', None,   'How were eviction orders treated during the COVID-19 period?'),
    (None,   None,   'What constitutes unconscionable conduct by a creditor under the NCA?'),
    (None,   None,   'When is a surety discharged from liability?'),
]

for year, type_f, q in tests:
    print(f'\n=== {q}')
    if year:     print(f'    Year filter: {year}')
    if type_f: print(f'    Type filter: {type_f}')
    res = semantic_search(q, k=3, year_filter=year, type_filter=type_f)
    try:
        from IPython.display import display
        display(res[['case_id','year','type','score','chunk_text']].head(3))
    except Exception:
        print(res[['case_id','year','type','score']].head(3).to_string(index=False))


=== What is the test for urgency in interdict applications?


,case_id,year,type,score,chunk_text
0,3342,2020,non-electoral,0.685195,e requirements\nfor the grant of an interim in...
1,3506,2021,non-electoral,0.673988,early trigger the interests of justice.\n\nThe...
2,3590,2021,non-electoral,0.666601,"examination), is a new phenomenon in our law, ..."



=== When will a court refuse to enforce a contractual penalty clause?


,case_id,year,type,score,chunk_text
0,1532,2008,non-electoral,0.731401,"n uncertainty. In matters of contract,\nfor ex..."
1,1360,2010,non-electoral,0.724536,iance with the clause or impossible\nfor the p...
2,1300,2010,non-electoral,0.721468,from acting on an unqualified\npromise to pay...



=== What are the requirements for a valid notice of appeal?


...

,case_id,year,type,score,chunk_text



=== What constitutes unconscionable conduct by a creditor under the NCA?


,case_id,year,type,score,chunk_text
0,2481,2014,non-electoral,0.719159,ulated limits is one. These exceptions are the...
1,1300,2010,non-electoral,0.717663,is correct:\nAustralian Competition and Consum...
2,4084,2023,non-electoral,0.703624,CMR.\n\n[3] The Regulator alleged that the sch...



=== When is a surety discharged from liability?


,case_id,year,type,score,chunk_text
0,2672,2014,non-electoral,0.703131,the bank to sue the\nsureties. The benefit of...
1,2672,2014,non-electoral,0.697410,creditor thereafter sets about executing the\...
2,2740,2012,non-electoral,0.697391,falsely implicated him.\n\nConclusion\n[50] It...


### B11. Experiments Dashboard

With the embedding model fixed, the remaining variables that shape retrieval quality are:

| Knob | What it controls | Trade-off |
|------|-----------------|-----------|
| `CHUNK_SIZE_VAR` | How much text surrounds each retrieved fact | Small: higher recall, noisier context; Large: coherent reasoning, lower precision |
| `YEAR_SCOPE` / `DIVISION_SCOPE` | Which sub-corpus is searched | Narrow: higher precision; Broad: higher recall |
| `TOP_K_VAR` | How many chunks are fed to the LLM | More: fewer missed facts; Fewer: less hallucination risk |
| `INDEX_SIZE_TARGET` | Simulated corpus size (latency test only) | FAISS stays sub-millisecond to ~100 K vectors |

**Insight:** once you have a strong embedding model, chunk size is usually the highest-leverage
parameter for legal text. Judgements have a clear paragraph structure -- a chunk boundary
that cuts across the *ratio decidendi* will reliably hurt recall regardless of the model.

In [23]:
#@title B11. Experiments Dashboard -- chunk size, scope, latency
#@markdown Adjust variables below and re-run.

CHUNK_SIZE_VAR  = 800    # @param {type:"integer"}
YEAR_SCOPE      = ""     # @param {type:"string"}
TYPE_SCOPE      = ""     # @param {type:"string"}
INDEX_SIZE_TARGET = 10000  # @param {type:"integer"}
QUERY_VAR = "What is the test for reasonable apprehension of bias?"  # @param {type:"string"}
TOP_K_VAR = 5  # @param {type:"integer"}

# 1) Rechunk
CHUNK_OVERLAP_VAR = max(50, int(0.20 * CHUNK_SIZE_VAR))
def _bld(df, size, overlap):
    rows = []
    for _, row in df.iterrows():
        t = str(row["document"])
        if not t.strip(): continue
        step = max(1, size - overlap); i = 0; j = 0
        while i < len(t):
            rows.append({"chunk_id": f"{row['doc_id']}-{j:04d}", "doc_id": row["doc_id"],
                         "case_id": row["case_id"], "year": row["year"],
                         "type":     row["type"], "chunk_text": t[i:i+size]})
            i += step; j += 1
    return pd.DataFrame(rows)

chunks_exp = _bld(corpus_df, CHUNK_SIZE_VAR, CHUNK_OVERLAP_VAR)
print(f"Rechunked: {len(chunks_exp)} chunks | size={CHUNK_SIZE_VAR} overlap={CHUNK_OVERLAP_VAR}")

# 2) Embed with bge-base-en-v1.5 (cache-aware)
emb_path_exp = os.path.join(SAVE_DIR, f"emb_cs{CHUNK_SIZE_VAR}_bge.npy")
chk_path_exp = os.path.join(SAVE_DIR, f"chunks_cs{CHUNK_SIZE_VAR}.parquet")
need = True
if os.path.isfile(emb_path_exp) and os.path.isfile(chk_path_exp):
    try:
        cached = pd.read_parquet(chk_path_exp)
        if len(cached) == len(chunks_exp):
            embs_exp = np.load(emb_path_exp); chunks_exp = cached; need = False
            print(f"Loaded cached: {emb_path_exp}")
    except: pass
if need:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer("BAAI/bge-base-en-v1.5")
    embs_exp = _m.encode(chunks_exp["chunk_text"].tolist(), batch_size=64,
                         show_progress_bar=True, normalize_embeddings=True,
                         convert_to_numpy=True).astype("float32")
    np.save(emb_path_exp, embs_exp)
    chunks_exp.to_parquet(chk_path_exp, index=False)

# 3) Index + retrieval
def _l2n(X): X = X.astype("float32"); return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
en = _l2n(embs_exp)
idx_exp = faiss.IndexFlatIP(en.shape[1]); idx_exp.add(en)

def _eq_bge(q):
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer("BAAI/bge-base-en-v1.5")
    return _l2n(m.encode([f"Represent this sentence: {q}"],
                          convert_to_numpy=True, normalize_embeddings=True).astype("float32"))

q_vec = _eq_bge(QUERY_VAR)
D, I  = idx_exp.search(q_vec, TOP_K_VAR)
res   = chunks_exp.iloc[I[0]][["case_id","year","type","chunk_text"]].copy()
res["score"] = D[0]
if YEAR_SCOPE.strip():     res = res[res["year"].astype(str) == YEAR_SCOPE]
if TYPE_SCOPE.strip(): res = res[res["type"].str.lower().str.contains(TYPE_SCOPE.lower(), na=False)]
print(f"Results (top-{TOP_K_VAR}):")
try:
    from IPython.display import display; display(res)
except: print(res.to_string(index=False))

# 4) Latency simulation
n_base  = len(embs_exp)
repeats = math.ceil(INDEX_SIZE_TARGET / n_base)
emb_big = _l2n(np.vstack([embs_exp] * repeats)[:INDEX_SIZE_TARGET])
idx_big = faiss.IndexFlatIP(emb_big.shape[1])
t0 = time.perf_counter(); idx_big.add(emb_big); t_build = time.perf_counter() - t0
R = 50; t0 = time.perf_counter()
for _ in range(R): idx_big.search(emb_big[:1], 5)
t_search = (time.perf_counter() - t0) / R
print(f"Latency @ {INDEX_SIZE_TARGET:,} vectors: build={t_build*1000:.1f}ms | search={t_search*1000:.3f}ms/query")
print(f"Summary: chunks={len(chunks_exp)} | top-k={TOP_K_VAR}")

Rechunked: 53053 chunks | size=800 overlap=160


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/829 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Results (top-5):


,case_id,year,type,chunk_text,score
27719,2216,2009,non-electoral,1 1999 (4) SA 147 (CC) para 48. This test was...,0.739904
27730,2216,2009,non-electoral,vassed before\nthe commencement of the hearing...,0.722906
27727,2216,2009,non-electoral,of the proceedings and that it was\ndi...,0.710033
22667,2678,2014,non-electoral,as follows:\n„When it is alleged that a defen...,0.709021
33037,3270,2007,non-electoral,agree. However bias is not\nper se to be infer...,0.708579


Latency @ 10,000 vectors: build=10.4ms | search=1.862ms/query
Summary: chunks=53053 | top-k=5


### B12. Try It Yourself -- Similar High-Demand South African Datasets

The RAG pipeline you built throughout this notebook is **dataset-agnostic** --
the same chunk -> embed -> FAISS -> generate flow works on any text corpus.
Below are three rich South African open datasets you can load and run through the same steps.

---

#### Dataset 1 -- COVID-19 ZA Health Data
**Source:** `https://github.com/dsfsi/covid19za`  
**Format:** CSV files (cases, deaths, tests, hospitalisations by province and date)  
**Suggested query:** *'Which provinces had the highest mortality rate during the second wave?'*

```python
import pandas as pd
df_covid = pd.read_csv(
    'https://raw.githubusercontent.com/dsfsi/covid19za/master/data/'
    'covid19za_provincial_cumulative_timeline_confirmed.csv'
)
df_covid['document'] = df_covid.apply(
    lambda r: ' | '.join(f'{c}: {r[c]}' for c in df_covid.columns), axis=1
)
# Then: chunks_df = build_chunks(df_covid) -> embed -> FAISS -> RAG
```

---

#### Dataset 2 -- DSFSI Financial / Local Datasets
**Source:** `https://github.com/dsfsi/dsfsi-datasets/blob/master/local`  
**Format:** Mixed CSV / text  
**Suggested query:** *'What financial inclusion trends are visible across South African municipalities?'*

```python
# Browse the repo index, then:
df_fin = pd.read_csv('your_chosen_file.csv')
df_fin['document'] = df_fin.astype(str).apply(' | '.join, axis=1)
# Run through Part B pipeline as-is
```

---

#### Dataset 3 -- South African News Corpus (2015-2019)
**Source:** `https://zenodo.org/record/3668495`  
**Format:** CSV (~200 K news articles)  
**Suggested query:** *'How was load-shedding covered before and after Eskom announcements?'*

```python
# After downloading from Zenodo:
df_news = pd.read_csv('sa_news.csv')
df_news['document'] = df_news['headline'] + ' ' + df_news['body']
# Run through Part B pipeline exactly as-is
```

---

### Running These on the AfriLink SDK

For any of the datasets above, once your corpus exceeds a few thousand rows,
embedding locally in Colab becomes a bottleneck. Submit the embedding step to
the SDK's HPC cluster for sovereign, high-throughput compute:

```python
from afrilink import AfriLinkClient
client = AfriLinkClient()
client.authenticate()

# Upload your corpus
client.upload_dataset('sa_news_corpus.jsonl', 'sa_news')

# The HPC cluster is co-located with your data and operates under
# industry-standard security and data residency controls --
# well-suited for health, financial, or legally sensitive datasets.
print(client.cert_minutes_remaining, 'minutes of session remaining')
```

The cluster returns your embeddings file (`.npy`) which drops straight into
the FAISS indexing cell above -- no other changes needed.

---

> **Challenge:** Pick one of the three datasets, load it, run all four Part A strategies
> on a single document, then build the full Part B pipeline.
> Experiment with chunk size, overlap, and metadata scope to find the settings that give
> the best precision for your chosen domain -- legal text, health time-series, and news
> articles each have different optimal chunking strategies.